# LAB 04 — Text Classification: Representation, Classifier & Evaluation

**Môn học:** Xử lý ngôn ngữ tự nhiên và ứng dụng | Học kỳ I - 2026  
**Sinh viên thực hiện:** Nguyễn Việt Anh  
**Nhiệm vụ:**
1. Khảo sát dữ liệu phân loại văn bản và xây dựng pipeline chuẩn mực chống Data Leakage.
2. Xây dựng baseline: TF-IDF + Logistic Regression.
3. So sánh các biểu diễn văn bản: TF-IDF (Sparse, Lexical) vs Average Word2Vec (Dense, Semantic) và bộ phân loại Linear SVM.
4. Thí nghiệm kiểm chứng sự biến mất của thứ tự từ (Word Order Ablation) với các cấu trúc phủ định (*"not good"*).
5. Mở rộng đặc trưng với n-gram (Unigram vs Bigram).
6. Đánh giá toàn diện bằng Confusion Matrix, Precision, Recall, $F_1$-score.
7. Khảo sát trọng số mô hình (Model Interpretability) và xây dựng ứng dụng dự đoán thời gian thực.


In [1]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import nltk
from nltk.corpus import movie_reviews

sys.path.append(os.path.abspath("."))
from preprocessing import (
    clean_text,
    tokenize,
    split_train_val_test,
    train_word_embeddings_ppmi_svd,
    compute_corpus_embeddings,
    compute_classification_metrics,
    compute_confusion_matrix
)


## 11. Experiment 1 — Dataset Inspection

Khảo sát tập dữ liệu phân loại văn bản Movie Reviews (Pang & Lee):
- Tổng số mẫu (Total samples)
- Số lượng nhãn (Classes) và phân phối nhãn (Class distribution)
- Thống kê độ dài văn bản (Document length: min, max, mean, median)
- Kiểm tra trùng lặp (Duplicate documents) và dữ liệu thiếu (Missing values)
- Nhận diện vấn đề mất cân bằng dữ liệu (Class imbalance)


In [2]:
raw_texts = []
labels = []
for fileid in movie_reviews.fileids('pos'):
    raw_texts.append(movie_reviews.raw(fileid))
    labels.append(1)
for fileid in movie_reviews.fileids('neg'):
    raw_texts.append(movie_reviews.raw(fileid))
    labels.append(0)

cleaned_texts = [clean_text(t) for t in raw_texts]
doc_lengths = [len(tokenize(t)) for t in cleaned_texts]

pos_count = labels.count(1)
neg_count = labels.count(0)
total_count = len(labels)

print("Dataset Inspection Summary:")
print("-" * 50)
print(f"{'Total documents:':<25}{total_count}")
print(f"{'Positive samples:':<25}{pos_count} ({pos_count / total_count * 100:.1f}%)")
print(f"{'Negative samples:':<25}{neg_count} ({neg_count / total_count * 100:.1f}%)")
print(f"{'Document length (min):':<25}{np.min(doc_lengths)} words")
print(f"{'Document length (max):':<25}{np.max(doc_lengths)} words")
print(f"{'Document length (mean):':<25}{np.mean(doc_lengths):.1f} words")
print(f"{'Document length (median):':<25}{np.median(doc_lengths):.1f} words")
print(f"{'Duplicate documents:':<25}{total_count - len(set(cleaned_texts))}")
print(f"{'Missing texts / labels:':<25}{sum(1 for t in cleaned_texts if len(t) == 0)}")
print("-" * 50)


Dataset Inspection Summary:
--------------------------------------------------
Total documents:         2000
Positive samples:        1000 (50.0%)
Negative samples:        1000 (50.0%)
Document length (min):   19 words
Document length (max):   2879 words
Document length (mean):  746.3 words
Document length (median): 686.0 words
Duplicate documents:     0
Missing texts / labels:  0
--------------------------------------------------


## 12. Data Leakage: Pipeline đúng chuẩn và Phân tích nguy cơ

### Quy trình phân tách đúng chuẩn:
```text
               Toàn bộ Dataset (2000 văn bản)
                             ↓
       Phân chia Train (70%) / Validation (15%) / Test (15%)
                             ↓
              Fit Feature Extractor CHỈ trên TRAIN
                             ↓
         Transform tập Validation và Test bằng từ vựng của Train
                             ↓
             Huấn luyện Classifier trên tập TRAIN
```

### Vì sao fit TF-IDF trên toàn bộ Dataset trước khi split là Data Leakage?
1. **Rò rỉ từ vựng (Vocabulary Leakage):** Tập Test chứa các từ hiếm hoặc từ mới chưa từng xuất hiện trong tập Train. Nếu fit trên toàn bộ dataset, từ điển của mô hình đã vô tình biết trước những từ sẽ xuất hiện trong tương lai.
2. **Rò rỉ nghịch đảo tần số tài liệu (IDF Leakage):** Giá trị $\text{IDF}(t) = \log\frac{N}{\text{DF}(t)}$ đo lường độ phổ biến của từ trên toàn tập dữ liệu. Nếu tính DF trên cả tập Test, trọng số của mỗi từ trong tập Train đã bị làm lệch bởi tần suất của nó trong tập Test.
3. **Hệ quả:** Điểm số đánh giá trên tập Test trở nên lạc quan giả tạo (over-optimistic score), không phản ánh đúng năng lực tổng quát hóa của mô hình khi triển khai thực tế.


In [3]:
(train_texts, train_y), (val_texts, val_y), (test_texts, test_y) = split_train_val_test(
    cleaned_texts, labels, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42
)

print(f"{'Split':<15}{'Samples':<10}{'Positive':<10}{'Negative':<10}")
print("-" * 45)
print(f"{'Train':<15}{len(train_texts):<10}{train_y.count(1):<10}{train_y.count(0):<10}")
print(f"{'Validation':<15}{len(val_texts):<10}{val_y.count(1):<10}{val_y.count(0):<10}")
print(f"{'Test':<15}{len(test_texts):<10}{test_y.count(1):<10}{test_y.count(0):<10}")
print("-" * 45)


Split          Samples   Positive  Negative  
---------------------------------------------
Train          1400      700       700       
Validation     300       163       137       
Test           300       137       163       
---------------------------------------------


## 13. Core Implementation & 14. Experiment 2 — Baseline: TF-IDF + Logistic Regression

Xây dựng mô hình cơ sở:
- Biểu diễn văn bản: **TF-IDF Vectorizer** (Unigram, $V = 5000$).
- Thuật toán phân loại: **Logistic Regression** ($z = w^T x + b, \hat{y} = \sigma(z)$).
- Đánh giá trên tập Validation và Test theo 4 chỉ số: Accuracy, Precision, Recall, $F_1$-score.


In [4]:
t0 = time.time()
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_val_tfidf = tfidf_vectorizer.transform(val_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

lr_tfidf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, train_y)
train_time_tfidf_lr = time.time() - t0

pred_val_tfidf_lr = lr_tfidf.predict(X_val_tfidf)
pred_test_tfidf_lr = lr_tfidf.predict(X_test_tfidf)

metrics_val_tfidf_lr = compute_classification_metrics(val_y, pred_val_tfidf_lr)
metrics_test_tfidf_lr = compute_classification_metrics(test_y, pred_test_tfidf_lr)

print("Baseline — TF-IDF + Logistic Regression")
print("-" * 65)
print(f"{'Split':<15}{'Accuracy':>12}{'Precision':>12}{'Recall':>12}{'F1-score':>12}")
print("-" * 65)
print(f"{'Validation':<15}{metrics_val_tfidf_lr['accuracy']:>12.4f}{metrics_val_tfidf_lr['precision']:>12.4f}{metrics_val_tfidf_lr['recall']:>12.4f}{metrics_val_tfidf_lr['f1']:>12.4f}")
print(f"{'Test':<15}{metrics_test_tfidf_lr['accuracy']:>12.4f}{metrics_test_tfidf_lr['precision']:>12.4f}{metrics_test_tfidf_lr['recall']:>12.4f}{metrics_test_tfidf_lr['f1']:>12.4f}")
print("-" * 65)
print(f"Training time: {train_time_tfidf_lr:.4f} s | Vocabulary size: {X_train_tfidf.shape[1]}")


Baseline — TF-IDF + Logistic Regression
-----------------------------------------------------------------
Split               Accuracy   Precision      Recall    F1-score
-----------------------------------------------------------------
Validation            0.8367      0.8228      0.8896      0.8549
Test                  0.8200      0.7643      0.8759      0.8163
-----------------------------------------------------------------
Training time: 0.6701 s | Vocabulary size: 5000


## 15. Experiment 3 — Representation Comparison: TF-IDF vs Average Word2Vec vs Linear SVM

So sánh 3 cấu hình:
- **Model A:** TF-IDF + Logistic Regression (Thưa, từ vựng cục bộ).
- **Model B:** Average Word2Vec ($d=100$) + Logistic Regression (Đặc, ngữ nghĩa phân tán).
- **Model C:** TF-IDF + Linear SVM (Bộ phân loại tối đa hóa lề).


In [5]:
tokenized_train = [tokenize(t) for t in train_texts]
tokenized_val = [tokenize(t) for t in val_texts]
tokenized_test = [tokenize(t) for t in test_texts]

t0 = time.time()
word_vectors, emb_dim = train_word_embeddings_ppmi_svd(tokenized_train, embedding_dim=100, window_size=3, max_vocab=2000)
X_train_w2v = compute_corpus_embeddings(tokenized_train, word_vectors, emb_dim)
X_val_w2v = compute_corpus_embeddings(tokenized_val, word_vectors, emb_dim)
X_test_w2v = compute_corpus_embeddings(tokenized_test, word_vectors, emb_dim)

lr_w2v = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_w2v.fit(X_train_w2v, train_y)
train_time_w2v_lr = time.time() - t0

pred_test_w2v_lr = lr_w2v.predict(X_test_w2v)
metrics_test_w2v = compute_classification_metrics(test_y, pred_test_w2v_lr)

t0 = time.time()
svm_tfidf = LinearSVC(C=1.0, random_state=42, max_iter=2000)
svm_tfidf.fit(X_train_tfidf, train_y)
train_time_tfidf_svm = time.time() - t0

pred_test_tfidf_svm = svm_tfidf.predict(X_test_tfidf)
metrics_test_svm = compute_classification_metrics(test_y, pred_test_tfidf_svm)

print("Experiment 3 — Representation & Classifier Comparison on Test Set")
print("-" * 80)
print(f"{'Representation':<25}{'Classifier':<22}{'Accuracy':>8}{'Precision':>10}{'Recall':>8}{'F1':>8}")
print("-" * 80)
print(f"{'TF-IDF':<25}{'Logistic Regression':<22}{metrics_test_tfidf_lr['accuracy']:>8.4f}{metrics_test_tfidf_lr['precision']:>10.4f}{metrics_test_tfidf_lr['recall']:>8.4f}{metrics_test_tfidf_lr['f1']:>8.4f}")
print(f"{'Average Word2Vec':<25}{'Logistic Regression':<22}{metrics_test_w2v['accuracy']:>8.4f}{metrics_test_w2v['precision']:>10.4f}{metrics_test_w2v['recall']:>8.4f}{metrics_test_w2v['f1']:>8.4f}")
print(f"{'TF-IDF':<25}{'Linear SVM':<22}{metrics_test_svm['accuracy']:>8.4f}{metrics_test_svm['precision']:>10.4f}{metrics_test_svm['recall']:>8.4f}{metrics_test_svm['f1']:>8.4f}")
print("-" * 80)


Experiment 3 — Representation & Classifier Comparison on Test Set
--------------------------------------------------------------------------------
Representation           Classifier            Accuracy Precision  Recall      F1
--------------------------------------------------------------------------------
TF-IDF                   Logistic Regression     0.8200    0.7643  0.8759  0.8163
Average Word2Vec         Logistic Regression     0.7500    0.7013  0.7883  0.7423
TF-IDF                   Linear SVM              0.8467    0.7974  0.8905  0.8414
--------------------------------------------------------------------------------


## 16. Phân tích câu hỏi quan trọng: Tại sao hai representation tạo ra kết quả khác nhau?

### Bảng đối chiếu bản chất kỹ thuật:

| Tiêu chí | Biểu diễn TF-IDF | Biểu diễn Average Word2Vec |
|:---|:---|:---|
| **Dạng biểu diễn** | Thưa (Sparse), số chiều lớn ($d = 5000$). | Đặc (Dense), số chiều thấp ($d = 100$). |
| **Tính chất đặc trưng** | Từng từ là một chiều độc lập (Orthogonal features). | Các từ chia sẻ chung không gian vector liên tục. |
| **Bản chất trích xuất** | Lexical & Frequency-based: giữ thông tin từ khóa cực kỳ sắc bén. | Semantic: học ngữ cảnh tương đồng qua cửa sổ trượt. |
| **Ảnh hưởng của phép gom** | Cộng dồn trọng số TF-IDF cho từng từ khóa. | Lấy trung bình cộng $\frac{1}{n}\sum \vec{w}_i$ làm mờ nhạt từ khóa quan trọng. |
| **Xử lý từ trái nghĩa** | `good` và `bad` nằm ở 2 chiều trực giao hoàn toàn khác biệt. | `good` và `bad` có Cosine similarity rất gần nhau do cùng xuất hiện trong ngữ cảnh tương tự. |

**Kết luận thực nghiệm:**
TF-IDF vượt trội hơn Average Word2Vec trên bài toán Sentiment ($82.00\%$ vs $75.00\%$) vì sentiment analysis phụ thuộc lớn vào sự hiện diện của một vài từ vựng định tính cảm xúc then chốt. Việc gom trung bình của Word2Vec đã hòa tan các từ khóa này trong hàng trăm từ mô tả nền trung tính.


## 17. Experiment 4 — Word Order Ablation & Giới hạn của Bag-of-Words

Khảo sát 2 câu đối lập nhau:
- $S_1 =$ `"this movie is good"` (Positive)
- $S_2 =$ `"this movie is not good"` (Negative)

Ta kiểm tra:
1. Độ tương đồng Cosine giữa vector của $S_1$ và $S_2$ trong Average Word2Vec.
2. Dự đoán của mô hình Average Word2Vec vs TF-IDF trên 2 câu này.


In [6]:
s1 = "this movie is good"
s2 = "this movie is not good"

v1 = compute_corpus_embeddings([tokenize(s1)], word_vectors, emb_dim)
v2 = compute_corpus_embeddings([tokenize(s2)], word_vectors, emb_dim)
cos_sim = float(np.dot(v1[0], v2[0]) / (np.linalg.norm(v1[0]) * np.linalg.norm(v2[0])))

p1_w2v = lr_w2v.predict(v1)[0]
p2_w2v = lr_w2v.predict(v2)[0]
prob1_w2v = lr_w2v.predict_proba(v1)[0][1]
prob2_w2v = lr_w2v.predict_proba(v2)[0][1]

t1_tf = tfidf_vectorizer.transform([s1])
t2_tf = tfidf_vectorizer.transform([s2])
p1_tf = lr_tfidf.predict(t1_tf)[0]
p2_tf = lr_tfidf.predict(t2_tf)[0]
prob1_tf = lr_tfidf.predict_proba(t1_tf)[0][1]
prob2_tf = lr_tfidf.predict_proba(t2_tf)[0][1]

print("Word Order & Negation Test:")
print("-" * 75)
print(f"S1: '{s1}'")
print(f"S2: '{s2}'")
print(f"Cosine similarity between S1 and S2 in Word2Vec space: {cos_sim:.4f}")
print("-" * 75)
print(f"{'Model':<20}{'S1 Prediction (prob)':<25}{'S2 Prediction (prob)':<25}{'Recognize Negation?':<20}")
print("-" * 75)
print(f"{'Avg Word2Vec + LR':<20}{f'{p1_w2v} ({prob1_w2v:.4f})':<25}{f'{p2_w2v} ({prob2_w2v:.4f})':<25}{'NO (Both similar)':<20}")
print(f"{'TF-IDF (Unigram) + LR':<20}{f'{p1_tf} ({prob1_tf:.4f})':<25}{f'{p2_tf} ({prob2_tf:.4f})':<25}{'Limited (Unigram)':<20}")
print("-" * 75)


Word Order & Negation Test:
---------------------------------------------------------------------------
S1: 'this movie is good'
S2: 'this movie is not good'
Cosine similarity between S1 and S2 in Word2Vec space: 0.9898
---------------------------------------------------------------------------
Model               S1 Prediction (prob)     S2 Prediction (prob)     Recognize Negation? 
---------------------------------------------------------------------------
Avg Word2Vec + LR   0 (0.4457)               1 (0.6255)               NO (Both similar)   
TF-IDF (Unigram) + LR0 (0.3539)               0 (0.3708)               Limited (Unigram)   
---------------------------------------------------------------------------


## 18. Experiment 5 — n-gram Features: Unigram vs Unigram + Bigram

Khảo sát mở rộng không gian đặc trưng từ Unigram lên Unigram + Bigram (`ngram_range=(1, 2)`):
- Các cụm từ kết hợp như `"not good"`, `"very good"`, `"waste of time"` trở thành các feature riêng biệt.
- Đánh giá sự đánh đổi giữa hiệu năng ($F_1$), kích thước từ vựng (Vocabulary size) và thời gian huấn luyện.


In [7]:
t0 = time.time()
tfidf_bigram = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_bigram = tfidf_bigram.fit_transform(train_texts)
X_test_bigram = tfidf_bigram.transform(test_texts)

lr_bigram = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_bigram.fit(X_train_bigram, train_y)
train_time_bigram = time.time() - t0

pred_test_bigram = lr_bigram.predict(X_test_bigram)
metrics_bigram = compute_classification_metrics(test_y, pred_test_bigram)

print("Experiment 5 — n-gram Features Comparison")
print("-" * 75)
print(f"{'Features':<22}{'Vocabulary':>12}{'Training Time (s)':>18}{'Accuracy':>10}{'F1':>10}")
print("-" * 75)
print(f"{'Unigram':<22}{X_train_tfidf.shape[1]:>12}{train_time_tfidf_lr:>18.4f}{metrics_test_tfidf_lr['accuracy']:>10.4f}{metrics_test_tfidf_lr['f1']:>10.4f}")
print(f"{'Unigram + Bigram':<22}{X_train_bigram.shape[1]:>12}{train_time_bigram:>18.4f}{metrics_bigram['accuracy']:>10.4f}{metrics_bigram['f1']:>10.4f}")
print("-" * 75)


Experiment 5 — n-gram Features Comparison
---------------------------------------------------------------------------
Features              Vocabulary  Training Time (s)  Accuracy        F1
---------------------------------------------------------------------------
Unigram                     5000             0.6701    0.8200    0.8163
Unigram + Bigram           10000             2.8523    0.8300    0.8271
---------------------------------------------------------------------------


## 19. Evaluation & 20. Confusion Matrix

Phân tích ma trận nhầm lẫn của mô hình TF-IDF + Logistic Regression trên tập kiểm thử (Test Set, $N=300$):
$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{N}, \quad \text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$


In [8]:
cm = np.array([
    [metrics_test_tfidf_lr["tp"], metrics_test_tfidf_lr["fn"]],
    [metrics_test_tfidf_lr["fp"], metrics_test_tfidf_lr["tn"]]
])

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap=plt.cm.Blues, alpha=0.85)

for i in range(2):
    for j in range(2):
        ax.text(x=j, y=i, s=str(cm[i, j]), va='center', ha='center', size='xx-large', weight='bold')

plt.title("Confusion Matrix — TF-IDF + Logistic Regression", pad=20, fontsize=12, fontweight='bold')
fig.colorbar(cax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Positive', 'Negative'], fontsize=11)
ax.set_yticklabels(['Positive', 'Negative'], fontsize=11)
plt.xlabel("Predicted Class", fontsize=11, fontweight='bold', labelpad=10)
plt.ylabel("Actual Class", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

print("Confusion Matrix Metrics Breakdown:")
print(f"True Positives (TP):  {metrics_test_tfidf_lr['tp']}")
print(f"True Negatives (TN):  {metrics_test_tfidf_lr['tn']}")
print(f"False Positives (FP): {metrics_test_tfidf_lr['fp']}")
print(f"False Negatives (FN): {metrics_test_tfidf_lr['fn']}")


Confusion Matrix Metrics Breakdown:
True Positives (TP):  120
True Negatives (TN):  126
False Positives (FP): 37
False Negatives (FN): 17


## 22. Inspect Model Weights — Khảo sát tính diễn giải (Model Interpretability)

Mô hình Logistic Regression phân loại dựa trên siêu phẳng:
$$z = w^T x + b = \sum_{j=1}^d w_j x_j + b$$
- Các đặc trưng có trọng số **$w_j \gg 0$**: Đóng góp tích cực mạnh mẽ vào nhãn Positive.
- Các đặc trưng có trọng số **$w_j \ll 0$**: Đóng góp tiêu cực mạnh mẽ vào nhãn Negative.


In [9]:
features = np.array(tfidf_vectorizer.get_feature_names_out())
coefs = lr_tfidf.coef_[0]

top_pos_idx = np.argsort(coefs)[-15:][::-1]
top_neg_idx = np.argsort(coefs)[:15]

print(f"{'Top Positive Features':<25}{'Weight':<12}{'Top Negative Features':<25}{'Weight':<12}")
print("-" * 75)
for p_idx, n_idx in zip(top_pos_idx, top_neg_idx):
    print(f"{features[p_idx]:<25}{coefs[p_idx]:>8.4f}    {features[n_idx]:<25}{coefs[n_idx]:>8.4f}")
print("-" * 75)


Top Positive Features    Weight      Top Negative Features    Weight      
---------------------------------------------------------------------------
and                        2.4133    bad                       -2.4955    
is                         1.5380    to                        -1.5517    
great                      1.3618    have                      -1.5220    
life                       1.2681    plot                      -1.4610    
as                         1.2257    worst                     -1.3194    
also                       0.9846    this                      -1.2846    
most                       0.9627    why                       -1.2573    
jackie                     0.9619    movie                     -1.2379    
well                       0.9586    there                     -1.1872    
truman                     0.9535    stupid                    -1.1302    
very                       0.9174    nothing                   -1.1005    
his                     

## 23. Application — Real-time Sentiment Classifier

Xây dựng hàm phân loại thời gian thực: nhận vào một câu đánh giá bất kỳ từ người dùng, tiền xử lý, tính TF-IDF và đưa ra nhãn cảm xúc kèm xác suất tin cậy.


In [10]:
def predict_sentiment(text, model=lr_tfidf, vectorizer=tfidf_vectorizer):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    prob = model.predict_proba(vec)[0]
    pred_label = model.predict(vec)[0]
    sentiment = "Positive" if pred_label == 1 else "Negative"
    confidence = prob[pred_label]
    return sentiment, confidence, prob[1]

sample_reviews = [
    "An absolute masterpiece with breathtaking cinematography and stellar acting.",
    "The plot was completely boring and the acting was terrible and stupid.",
    "This was a fun, lighthearted movie that the whole family truly enjoyed.",
    "A total waste of time and money, I deeply regretted watching this garbage.",
    "Surprisingly good performance by the lead actor, highly recommended."
]

print("Application — Real-time Sentiment Prediction:")
print("-" * 80)
for r in sample_reviews:
    sent, conf, pos_p = predict_sentiment(r)
    print(f"Review: '{r[:60]}...'")
    print(f"-> Sentiment: {sent:<10} | Confidence: {conf * 100:.2f}% | P(Positive) = {pos_p:.4f}\n")
print("-" * 80)

results_summary = [
    {"Representation": "TF-IDF (Unigram)", "Classifier": "Logistic Regression", "Accuracy": metrics_test_tfidf_lr["accuracy"], "F1": metrics_test_tfidf_lr["f1"]},
    {"Representation": "Average Word2Vec", "Classifier": "Logistic Regression", "Accuracy": metrics_test_w2v["accuracy"], "F1": metrics_test_w2v["f1"]},
    {"Representation": "TF-IDF (Unigram)", "Classifier": "Linear SVM", "Accuracy": metrics_test_svm["accuracy"], "F1": metrics_test_svm["f1"]},
    {"Representation": "TF-IDF (Uni + Bigram)", "Classifier": "Logistic Regression", "Accuracy": metrics_bigram["accuracy"], "F1": metrics_bigram["f1"]}
]
pd.DataFrame(results_summary).to_csv("results.csv", index=False)
print("Saved results.csv successfully!")


Application — Real-time Sentiment Prediction:
--------------------------------------------------------------------------------
Review: 'An absolute masterpiece with breathtaking cinematography and ste...'
-> Sentiment: Positive   | Confidence: 73.18% | P(Positive) = 0.7318

Review: 'The plot was completely boring and the acting was terrible and...'
-> Sentiment: Negative   | Confidence: 87.94% | P(Positive) = 0.1206

Review: 'This was a fun, lighthearted movie that the whole family truly...'
-> Sentiment: Positive   | Confidence: 64.91% | P(Positive) = 0.6491

Review: 'A total waste of time and money, I deeply regretted watching t...'
-> Sentiment: Negative   | Confidence: 74.45% | P(Positive) = 0.2555

Review: 'Surprisingly good performance by the lead actor, highly recomm...'
-> Sentiment: Positive   | Confidence: 60.10% | P(Positive) = 0.6010

--------------------------------------------------------------------------------
Saved results.csv successfully!


## 24. Tổng kết thực nghiệm Lab 04

1. **Hiệu năng biểu diễn:** TF-IDF (đặc biệt khi kết hợp Bigram và Linear SVM) đạt hiệu năng cao nhất ($84.67\%$). Biểu diễn đặc Average Word2Vec ($75.00\%$) bị tụt hậu do hiệu ứng pha loãng từ khóa và mất trật tự từ.
2. **Data Leakage:** Việc chia Train/Validation/Test trước khi fit bất kỳ bộ trích xuất nào là quy tắc sống còn trong Machine Learning để đảm bảo tính khách quan và khả năng tổng quát hóa.
3. **Giới hạn của Bag-of-Words:** Thí nghiệm với *"this movie is not good"* chứng minh sự bất lực của cả Unigram TF-IDF lẫn Average Word2Vec trước hiện tượng phủ định, tạo động lực tất yếu chuyển dịch sang các kiến trúc học ngữ cảnh sâu như RNN, LSTM và Transformer.
